# Lighteval Example with Eval Hub SDK

This notebook demonstrates how to use the EvalHub client SDK with the EvalHub evaluation service. This notebook uses `SyncEvalHubClient`.

## Prerequisites: EvalHub credentials (`.env`)

Before running this notebook, create a `.env` file with your EvalHub credentials. Run the block below in a terminal (e.g. from the `examples/` directory); it writes `EVALHUB_TOKEN`, `EVALHUB_URL` and `EVALHUB_TENANT` to `.env`. The notebook will use these when it loads the client (e.g. via `load_dotenv()`).

EvalHub now runs in Multi Tenancy mode, and requires additional admin steps . Please see https://eval-hub.github.io/development/multi-tenancy for details. It requires creation of namespace, role and rolebinding. Both ServiceAccount Token and User Token are supported. This notebook uses User Token

```bash
# Get your OpenShift authentication token
export EVALHUB_TOKEN=$(oc whoami -t)

# Get the EvalHub route URL from your namespace (e.g., 'evalhub-test')
export EVALHUB_URL=https://$(oc get route evalhub -n evalhub-test -o jsonpath='{.spec.host}')

# The example uses "team-a" as its Tenant. ie. the namespace where the k8s jobs will run
export EVALHUB_TENANT="team-a"

# Write to .env file
cat > .env << EOF
EVALHUB_TOKEN=$EVALHUB_TOKEN
EVALHUB_URL=$EVALHUB_URL
EVALHUB_TENANT=$EVALHUB_TENANT
EOF

# Verify
echo "Token: ${EVALHUB_TOKEN:0:20}..."
echo "URL: $EVALHUB_URL"
echo "Tenant: $EVALHUB_TENANT"
echo "Written to .env"
```


## OpenShift Authentication

The SDK has built-in support for OpenShift authentication with three methods:

1. **ServiceAccount Token** (`auth_token_path`): For pods running inside OpenShift, use the automounted token at `/var/run/secrets/kubernetes.io/serviceaccount/token`
2. **User Token** (`auth_token`): For local development, use token from `oc whoami -t` (shown above)
3. **Environment Variable** (`auth_token`): For production, use token from environment variable or secret (used in examples below)


In [1]:
# Read environment variables from .env if present (for local runs)
from dotenv import load_dotenv
load_dotenv(".env")

True

## Setup: Import Required Modules

In [2]:
from evalhub import (
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
    JobStatus,
)
import os

# Verify environment variables are set
evalhub_token = os.getenv("EVALHUB_TOKEN")
if not evalhub_token:
    raise ValueError(
        "EVALHUB_TOKEN environment variable is not set. See Prerequisites section above."
    )

evalhub_url = os.getenv("EVALHUB_URL")
if not evalhub_url:
    raise ValueError(
        "EVALHUB_URL environment variable is not set. See Prerequisites section above."
    )

evalhub_tenant = os.getenv("EVALHUB_TENANT")
if not evalhub_tenant:
    raise ValueError(
        "EVALHUB_TENANT environment variable is not set. See Prerequisites section above."
    )

## Connect to EvalHub and check health status.

In [3]:
client = SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=evalhub_token,
    insecure=True,
    tenant=evalhub_tenant,
)
try:
    # Check health
    health = client.health()
    print(f"✓ EvalHub is healthy: {health['status']}")
    print(f"  Version: {health.get('version', 'unknown')}")
    print(f"  Uptime: {health.get('uptime_seconds', 0):.1f}s")
except Exception as e:
    print(f"✗ Failed to connect: {e}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ EvalHub is healthy: healthy
  Version: unknown
  Uptime: 0.0s


## List Providers and Benchmarks

In [4]:
# List available providers using nested resource structure
# List all providers
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

✓ Found 4 providers:
  - lm_evaluation_harness: LM Evaluation Harness
  - lighteval: Lighteval
  - guidellm: GuideLLM
  - garak: Garak


### We are using lighteval provider_id for this notebook
List available benchmarks for the provider

In [5]:
provider_id = "lighteval"

In [6]:
benchmarks = client.benchmarks.list(provider_id=provider_id)
print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
for benchmark in benchmarks:
    print(f"  - {benchmark.id}: {benchmark.name}")


✓ Found 23 benchmarks for lighteval:
  - commonsense_reasoning: Commonsense Reasoning Suite
  - scientific_reasoning: Scientific Reasoning Suite
  - physical_commonsense: Physical Commonsense Suite
  - truthfulness: Truthfulness Suite
  - math: Math Suite
  - knowledge: Knowledge Suite
  - language_understanding: Language Understanding Suite
  - hellaswag: HellaSwag
  - winogrande: Winogrande
  - openbookqa: OpenBookQA
  - arc:easy: ARC Easy
  - arc:challenge: ARC Challenge
  - piqa: PIQA
  - truthfulqa:mc: TruthfulQA MC
  - truthfulqa:generation: TruthfulQA Generation
  - gsm8k: GSM8K
  - math:algebra: MATH Algebra
  - math:counting_and_probability: MATH Counting & Probability
  - mmlu: MMLU
  - triviaqa: TriviaQA
  - glue:cola: GLUE CoLA
  - glue:sst2: GLUE SST-2
  - glue:mrpc: GLUE MRPC


## Submit an Evaluation Job

To find available models in your namespace:

```bash
# List services in your namespace
oc get svc -n <namespace>

# Example: Using a service named 'vllm-llama3-8b-instruct-svc' running in the 'evalhub-test' namespace on port 8000.
# The model endpoint URL is formatted as:
#   http://<service-name>.<namespace>.svc.cluster.local:<port>/v1
#
# Correct URL format: http://vllm-llama3-8b-instruct-svc.evalhub-test.svc.cluster.local:8000/v1
# WRONG format: http://vllm-llama3-8b-instruct-svc.evalhub-test.svc.cluster.local:8000/v1/completions
```

In [7]:
model_url = "http://vllm-llama3-8b-instruct-svc.evalhub-test.svc.cluster.local:8000/v1"
model_name = "meta-llama/Llama-3.1-8B-Instruct"
job_name = "ev-job1"
benchmark_id = "gsm8k"

In [8]:
# Create job submission request
job_request = JobSubmissionRequest(
    name=job_name,
    model=ModelConfig(url=model_url, name=model_name),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters={
                "num_examples": 30,  # Number of examples to evaluate (use small number for testing)
                "num_fewshot": 2,
            },
        )
    ]
)

# Submit job using nested resource
job = client.jobs.submit(job_request)

# Store job ID for later use
submitted_job_id = job.id
print(f"✓ Job submitted successfully")
print(f"  Job ID: {submitted_job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}")

✓ Job submitted successfully
  Job ID: 8b50e890-02e9-4738-9316-ea1b4bedc978
  State: JobStatus.PENDING
  Created: 2026-03-31 10:47:57.429440+00:00


In [9]:
# Check status
updated_job = client.jobs.get(submitted_job_id)
print(f"\n✓ Current job state: {updated_job.state}")
if updated_job.status and updated_job.status.message:
    print(f"  Message: {updated_job.status.message.message}")


✓ Current job state: JobStatus.RUNNING
  Message: Evaluation job is running


## Wait for the job to complete

In [10]:
import time

while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(submitted_job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print(f"✓ Job completed successfully")


✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING
✓ Job completed successfully


You can check the benchmark metrics from the job results

In [11]:
print("benchmark metrics:", updated_job.results.benchmarks[0].metrics)

benchmark metrics: {'all.extractive_match': 0, 'gsm8k|0.extractive_match': 0}


In [12]:
# print the job details
print(updated_job.model_dump_json(indent=2))

{
  "resource": {
    "id": "8b50e890-02e9-4738-9316-ea1b4bedc978",
    "tenant": "team-a",
    "created_at": "2026-03-31T10:47:57Z",
    "updated_at": "2026-03-31T10:48:26Z",
    "mlflow_experiment_id": null,
    "message": null
  },
  "status": {
    "state": "completed",
    "message": {
      "message": "Evaluation job is completed",
      "message_code": "evaluation_job_updated"
    },
    "benchmarks": [
      {
        "id": "gsm8k",
        "provider_id": "lighteval",
        "benchmark_index": 0,
        "status": "completed",
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-03-31T10:48:26.518365Z"
      }
    ]
  },
  "results": {
    "benchmarks": [
      {
        "id": "gsm8k",
        "provider_id": "lighteval",
        "benchmark_index": 0,
        "metrics": {
          "all.extractive_match": 0,
          "gsm8k|0.extractive_match": 0
        },
        "artifacts": {},
        "mlflow_run_id": null,
        "logs_path": null
   

## Clean up

In [13]:
# run this cell to delete the job with hard_delete=True
client.jobs.cancel(submitted_job_id, hard_delete=True)

True

In [14]:
## Close the client
client.close()